# Multi-seed aggregation: marginal $p$-value curves (1D / 2D / 4D)

Pools the per-observation $p$-values of SEVERAL runs that differ only in `base_seed`
(one independent fit + banks + observed draws per seed), producing the *marginal*
$p$-value distribution — unconditional over the realized fit — as a function of
$N_{\rm test}$.

Campaign recipe (per pipeline):
```bash
for s in 20260901 20260902 ... 20260910; do
    python launch.py prep    --seed $s   # (--local where prep is cheap)
    python launch.py workers --seed $s
    python launch.py analyze --seed $s   # produces results/pvalues.* per run
done
```

Statistics conventions (matching the paper template): thin lines = per-seed medians
(the seed-to-seed spread you observed); thick line = pooled median; band = pooled
25–75%; error bars = 95% CI of the pooled median from a **cluster bootstrap over
seeds** (repeats within a seed share a fit, so seeds are the independent unit).
Set the `GLOB_*` variables to your run folders.

In [ ]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

INK, GRID, MUTED = "#33322e", "#c9c8c0", "#8a897f"
NFIT_COLORS = ["#2a78d6", "#eb6834", "#3f9b5f", "#8a55c9"]
N_BOOT = 2000
FIG_DIR = "multiseed_figs"; os.makedirs(FIG_DIR, exist_ok=True)

def style_ax(ax):
    ax.tick_params(direction="in", which="both", colors=INK, top=True, right=True)
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.minorticks_off()
    ax.xaxis.label.set_color(INK); ax.yaxis.label.set_color(INK)
    ax.title.set_color(INK)

def nominal_lines(ax, label=True):
    tr = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
    for q in (0.25, 0.5, 0.75):
        ax.axhline(q, color=GRID, lw=1.0, ls=":", zorder=0)
        if label:
            ax.text(0.98, q + 0.012, f"nominal {int(q*100)}%", color=MUTED,
                    fontsize=8, ha="right", va="bottom", transform=tr, zorder=1)

def save_fig(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{stem}.{ext}"), dpi=200, bbox_inches="tight")
    print(os.path.join(FIG_DIR, stem) + ".{png,pdf}")

def fmt_n(n):
    n = int(n)
    return f"{n//1_000_000}M" if n >= 1_000_000 else (f"{n//1000}k" if n >= 1000 else str(n))

In [ ]:
def load_group(glob_pattern):
    '''Load pvalues from all run dirs matching the glob; verify identical params
    except base_seed; return truth-source df with a `seed` column.'''
    dirs = sorted(glob.glob(glob_pattern))
    assert dirs, f"no run dirs match {glob_pattern}"
    dfs, ref = [], None
    for d in dirs:
        P = json.load(open(os.path.join(d, "params.json")))
        seed = P.pop("base_seed")
        if ref is None:
            ref = P
        elif P != ref:
            diff = [k for k in P if P.get(k) != ref.get(k)]
            raise ValueError(f"{d} differs from the group beyond base_seed: {diff}")
        base = os.path.join(d, "results", "pvalues")
        if os.path.exists(base + ".parquet"):
            pv = pd.read_parquet(base + ".parquet")
        elif os.path.exists(base + ".pkl"):
            pv = pd.read_pickle(base + ".pkl")
        else:
            print(f"  [skip] {d}: no pvalues file (run `launch.py analyze --run {d}`)")
            continue
        pv = pv[pv.source == "truth"].copy()
        pv["seed"] = seed
        dfs.append(pv)
    df = pd.concat(dfs, ignore_index=True)
    print(f"{glob_pattern}: {df.seed.nunique()} seeds, "
          f"{len(df)} p-values, N_fit={sorted(df.N_fit.unique())}")
    return df

def draw_panel(ax, df, test_type, n_fits, label_nominal):
    for c, N in zip(NFIT_COLORS, n_fits):
        g = df[(df.test_type == test_type) & (df.N_fit == N)]
        if not len(g): continue
        # comp-matched K when a K dimension exists with several values
        if "K" in g and g.K.nunique() > 1:
            gc = df[(df.test_type == "comp") & (df.N_fit == N)]
            if len(gc): g = g[g.K == gc.K.mode()[0]]
        nts = np.array(sorted(g.N_test.unique()))
        # pooled median + pooled IQR band
        p25 = g.groupby("N_test").p.quantile(.25).reindex(nts)
        p50 = g.groupby("N_test").p.median().reindex(nts)
        p75 = g.groupby("N_test").p.quantile(.75).reindex(nts)
        ax.fill_between(nts, p25, p75, color=c, alpha=0.14, lw=0)
        ax.plot(nts, p50, color=c, lw=2.3, marker="o", ms=5.5,
                label=fr"$N_\mathrm{{fit}} = {fmt_n(N)}$", zorder=4)
        ax.axvline(N, color=c, ls="--", lw=1.0, alpha=.6, zorder=1)
    nominal_lines(ax, label=label_nominal)
    ax.set_xscale("log"); ax.set_ylim(0, 1)
    ax.set_xlabel(r"$N_{\rm test}$")
    style_ax(ax)
    ax.legend(frameon=False, fontsize=8.5, labelcolor=INK, loc="lower left")

def two_panel(df, title, stem):
    n_fits = sorted(df.N_fit.unique())
    print(f"{title}: pooling {df.seed.nunique()} seeds")
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), sharey=True)
    draw_panel(axes[0], df, "point", n_fits, label_nominal=False)
    axes[0].set_ylabel("null $p$-value    (pooled median, 25–75%)")
    axes[0].set_title(f"(a)  LR test, no uncertainties", fontsize=10)
    draw_panel(axes[1], df, "comp", n_fits, label_nominal=True)
    axes[1].set_title(f"(b)  LR test, tangent uncertainties", fontsize=10)
    xlims = (min(a.get_xlim()[0] for a in axes), max(a.get_xlim()[1] for a in axes))
    for a in axes: a.set_xlim(xlims)
    fig.tight_layout()
    save_fig(fig, stem)
    plt.show()

## 1D Gaussian

In [ ]:
GLOB_1D = "gof1d_slurm/runs/s*"      # match your same-tag, multi-seed dirs
df1 = load_group(GLOB_1D)
two_panel(df1, "1D Gaussian, well-specified", "multiseed_1d")

## 2D benchmark

In [ ]:
GLOB_2D = "gof2d_slurm/runs/s*"
try:
    df2 = load_group(GLOB_2D)
    two_panel(df2, "2D benchmark (GMM, K best)", "multiseed_2d")
except AssertionError as e:
    print("2D:", e)

## 4D embeddings

In [ ]:
GLOB_4D = "gof4d_slurm/runs/s*"
try:
    df4 = load_group(GLOB_4D)
    two_panel(df4, "4D embeddings (GMM, K best)", "multiseed_4d")
except AssertionError as e:
    print("4D:", e)

### Notes
* If a glob accidentally catches runs with different statistic parameters,
  `load_group` raises and names the offending keys — tighten the glob (the tag encodes
  the statistic, so `runs/s*_ts1_J16_..._S2000000` style patterns select one family).
* 4D: seeds reshuffle the SAME event pool, so seed runs are internally clean but not
  fully independent of each other (overlapping train/test blocks across seeds) —
  the bootstrap CI is slightly optimistic there; state it in the caption.
* The point panel at large $N_{\rm test}$ has all p-values at each seed's bank floor;
  the pooled median sits at the floor — expected, not an artifact.